In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [15]:
import subprocess; print(subprocess.check_output(['nvidia-smi']).decode('utf-8'))

Sun Apr 19 15:34:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.58.03              Driver Version: 595.58.03      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-PCIE-40GB          On  |   00000000:37:00.0 Off |                    0 |
| N/A   69C    P0             54W /  250W |     612MiB /  40960MiB |     25%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

# Load Dataset

In [2]:
import random
from torch.utils.data import Subset
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

data_dir = "./imagenet_validation"      # Download from https://www.kaggle.com/datasets/tusonggao/imagenet-validation-dataset

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
])

dataset = datasets.ImageFolder(root=data_dir, transform=transform)

n_samples = 5000
# Set seed for reproducibility
random.seed(42)
indices = random.sample(range(len(dataset)), n_samples)
subset = Subset(dataset, indices)

loader = DataLoader(
    subset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
)

num_classes = len(dataset.classes)
print("Dataset size:", len(loader.dataset))
print("Classes:", num_classes)

Dataset size: 5000
Classes: 1000


In [3]:
# Create a training DataLoader using the complementary indices (invert selected indices)
all_indices = set(range(len(dataset)))
selected_set = set(indices)
train_indices = list(all_indices - selected_set)
train_indices.sort()
train_subset = Subset(dataset, train_indices)
train_loader = DataLoader(train_subset, batch_size=64, shuffle=True, num_workers=4)
print("Validation dataset size:", len(loader.dataset))
print("Training dataset size:", len(train_loader.dataset))

Validation dataset size: 5000
Training dataset size: 45000


# Canonicalization

In [7]:
import timm
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF


class Canonicalizer(nn.Module):
    """
    Lightweight MobileNetV3-Small regressor that predicts the applied rotation
    angle (in degrees) and inverts it before passing to the frozen ViT.
    """
    def __init__(self, device):
        super().__init__()
        backbone = timm.create_model("efficientnet_b0", pretrained=True)
        in_features = backbone.classifier.in_features
        backbone.classifier = nn.Linear(in_features, 1)  # predict scalar angle
        self.regressor = backbone
        self.device = device

    def forward(self, x):
        theta = self.regressor(x).squeeze(1)          # [B]
        # Invert: rotate each image by -theta to restore canonical orientation
        canonical = torch.stack([
            TF.rotate(x[i].unsqueeze(0), -theta[i].item()).squeeze(0)
            for i in range(x.size(0))
        ])
        return canonical, theta

In [8]:
from tqdm import tqdm


def angle_loss(predicted_deg, gt_deg):
    # Convert to radians
    pred_rad = torch.deg2rad(predicted_deg)
    gt_rad   = torch.deg2rad(gt_deg)

    # Compare on unit circle — handles wraparound naturally
    loss = (torch.sin(pred_rad) - torch.sin(gt_rad)).pow(2) + \
           (torch.cos(pred_rad) - torch.cos(gt_rad)).pow(2)
    return loss.mean()


def train_canonicalizer(canonicalizer, loader, device, epochs=10, lr=1e-3, save_name=None):
    canonicalizer.train()
    optimizer = torch.optim.Adam(canonicalizer.parameters(), lr=lr)

    for epoch in range(epochs):
        total_loss = 0.0
        for images, _ in tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}"):
            images = images.to(device)
            B = images.size(0)

            # Sample a different random angle for each image in the batch
            gt_angles = torch.FloatTensor(B).uniform_(-180, 180).to(device)

            # Apply per-image rotation
            rotated = torch.stack([
                TF.rotate(images[i].unsqueeze(0), gt_angles[i].item()).squeeze(0)
                for i in range(B)
            ])

            _, predicted_angles = canonicalizer(rotated)
            loss = angle_loss(predicted_angles, gt_angles)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1} | MSE loss: {total_loss / len(loader):.4f}")

    if save_name is not None:
        # Save the trained canonicalizer for later use
        torch.save(canonicalizer.state_dict(), save_name)
        print(f"Canonicalizer model saved to {save_name}")

    return canonicalizer

In [9]:
import os

canonicalizer = Canonicalizer(device).to(device)

# if os.path.exists("canonicalizer.pth"):
#     canonicalizer.load_state_dict(torch.load("canonicalizer.pth", map_location=device))
#     print("Loaded canonicalizer from checkpoint.")

train_canonicalizer(canonicalizer, train_loader, device, epochs=10, lr=1e-3, save_name="efficientnetb0_canonicalizer.pth")

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Epoch 1/10: 100%|█████████████████████████████████████████████████████████████████████| 704/704 [01:56<00:00,  6.06it/s]


Epoch 1 | MSE loss: 1.7157


Epoch 2/10: 100%|█████████████████████████████████████████████████████████████████████| 704/704 [01:50<00:00,  6.34it/s]


Epoch 2 | MSE loss: 1.7426


...

Epoch 9/10: 100%|█████████████████████████████████████████████████████████████████████| 704/704 [01:30<00:00,  7.77it/s]


Epoch 9 | MSE loss: 1.5845


Epoch 10/10: 100%|████████████████████████████████████████████████████████████████████| 704/704 [01:32<00:00,  7.64it/s]


Epoch 10 | MSE loss: 1.4877
Canonicalizer model saved to efficientnetb0_canonicalizer.pth


Canonicalizer(
  (regressor): EfficientNet(
    (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
          )
          (conv_pw)

# Evaluate Models

In [11]:
import timm
import torch.nn as nn


class ModelWrapper(nn.Module):
    def __init__(self, model_name, num_classes, device):
        super().__init__()
        self.name = model_name
        self.device = device

        # ===== ViT (timm supervised) =====
        if model_name == "vit_base":
            self.model = timm.create_model(
                "vit_base_patch16_224", pretrained=True, num_classes=num_classes
            )
        elif model_name == "vit_large":
            self.model = timm.create_model(
                "vit_large_patch16_224", pretrained=True, num_classes=num_classes
            )

        # ===== DINOv2 =====
        elif model_name == "dinov2_base":
            self.model = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14_lc")
        elif model_name == "dinov2_large":
            self.model = torch.hub.load("facebookresearch/dinov2", "dinov2_vitl14_lc")

        else:
            raise ValueError(f"Unknown model: {model_name}")

        self.model = self.model.to(device)
        self.eval()

    def forward(self, images):
        with torch.no_grad():
            return self.model(images)

In [12]:
from pathlib import Path
from tqdm import tqdm
import time
import torchvision.transforms.functional as TF
import numpy as np


def apply_affine_batch(images, angle, translate, scale, shear):
    return torch.stack(
        [
            TF.affine(img, angle=angle, translate=translate, scale=scale, shear=shear)
            for img in images
        ]
    )


def extract_features(model, x):
    """Extract CLS token from timm ViT before the classification head."""
    # timm's forward_features returns the full encoder output
    # For ViT, index 0 is the CLS token
    return model.model.forward_features(x)[:, 0]


def evaluate(
    model,
    loader,
    model_name: str = "Model",
    translation: tuple[float, float] = [0, 0],
    rotation: float = 0.0,
    scale: float = 1.0,
    shear: tuple[float, float] = [0, 0],
    save_results_path: str = "improved_results.csv",
    improvement_type: str = "None",
    n_augmentations: int = 0,
):
    correct_top1 = 0
    correct_top5 = 0
    total = 0
    tta_angles = np.linspace(0, 360, num=n_augmentations+1, endpoint=False)

    start_time = time.time()

    with torch.no_grad():
        for images, labels in tqdm(loader):
            images = apply_affine_batch(
                images, angle=rotation, translate=translation, scale=scale, shear=shear
            )

            images = images.to(device)
            labels = labels.to(device)

            # TTA: average logits over rotations
            if improvement_type == "TTA":
                logits_sum = torch.zeros(images.size(0), num_classes, device=device)
                for angle in tta_angles:
                    rotated = TF.rotate(images, angle)
                    logits_sum += model(rotated)

                outputs = logits_sum / len(tta_angles)
            # Feature averaging: average CLS token features over rotations, then apply head once
            elif improvement_type == "feature_averaging":
                features_sum = None
                for angle in tta_angles:
                    rotated = TF.rotate(images, angle)
                    feats = extract_features(model, rotated)
                    if features_sum is None:
                        features_sum = feats
                    else:
                        features_sum += feats

                avg_features = features_sum / len(tta_angles)
                outputs = model.model.head(avg_features)
            # Predict and invert the rotation, then classify once
            elif improvement_type == "canonicalizer":
                canonical_images, _ = canonicalizer(images)
                outputs = model(canonical_images)
            else:
                outputs = model(images)

            preds_top1 = outputs.argmax(dim=1)
            correct_top1 += (preds_top1 == labels).sum().item()

            _, preds_top5 = outputs.topk(5, dim=1)
            correct_top5 += (preds_top5 == labels.unsqueeze(1)).any(dim=1).sum().item()

            total += labels.size(0)

    end_time = time.time()
    eval_time = end_time - start_time

    acc_top1 = correct_top1 / total
    acc_top5 = correct_top5 / total

    results_path = Path(save_results_path)
    header = (
        "Model Name,Dataset Size,Translation X, Translation Y,"
        "Rotation,Scale,Shear X, Shear Y,Top-1 Accuracy,Top-5 Accuracy,Type,Time"
    )

    if not results_path.exists() or results_path.stat().st_size == 0:
        # file doesn't exist or is empty -> create with header
        with results_path.open("w", encoding="utf-8") as f:
            f.write(header + "\n")
    else:
        # file exists, check header consistency
        with results_path.open("r", encoding="utf-8") as f:
            first_line = f.readline().strip()
        if first_line != header:
            with results_path.open("w", encoding="utf-8") as f:
                f.write(header + "\n")

    # append result row
    with results_path.open("a", encoding="utf-8") as f:
        improvement_type = f"{improvement_type}_{n_augmentations}" if n_augmentations > 0 else improvement_type
        f.write(
            f"{model_name},{len(loader.dataset)},"
            f"{translation[0]},{translation[1]},{rotation},{scale},"
            f"{shear[0]},{shear[1]},{acc_top1:.4f},{acc_top5:.4f},{improvement_type},{eval_time:.2f}\n"
        )

    return acc_top1, acc_top5

In [13]:
def evaluate_architecture(model_name, loader, geometric_transforms, improvement_type="None", n_augmentations=0):
    model = ModelWrapper(model_name, num_classes, device)

    # Baseline evaluation without transformations
    if geometric_transforms["baseline"]:
        print(f"Baseline (No Transformations):")
        acc_top1, acc_top5 = evaluate(model, loader, model_name, improvement_type=improvement_type, n_augmentations=n_augmentations)
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

    # Translations
    for translation in geometric_transforms["translations"]:
        print(f"Translation {translation}:")
        acc_top1, acc_top5 = evaluate(model, loader, model_name, translation=translation, improvement_type=improvement_type, n_augmentations=n_augmentations)
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

    # Rotations
    for rotation in geometric_transforms["rotations"]:
        print(f"Rotation {rotation}:")
        acc_top1, acc_top5 = evaluate(model, loader, model_name, rotation=rotation, improvement_type=improvement_type, n_augmentations=n_augmentations)
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

    # Scales
    for scale in geometric_transforms["scales"]:
        print(f"Scale {scale}:")
        acc_top1, acc_top5 = evaluate(model, loader, model_name, scale=scale, improvement_type=improvement_type, n_augmentations=n_augmentations)
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

    # Shears
    for shear in geometric_transforms["shears"]:
        print(f"Shear {shear}:")
        acc_top1, acc_top5 = evaluate(model, loader, model_name, shear=shear, improvement_type=improvement_type, n_augmentations=n_augmentations)
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

### ViT Base

##### 1. TTA

In [28]:
geometric_transforms = {
    "baseline": True,
    "translations": [
    #     (25, 0),  # Translate 25 pixels along X-axis
    #     (0, 25),  # Translate 25 pixels along Y-axis
    #     (-25, 0),  # Translate -25 pixels along X-axis
    #     (0, -25),  # Translate -25 pixels along Y-axis
    #     (50, 0),  # Translate 50 pixels along X-axis
    #     (0, 50),  # Translate 50 pixels along Y-axis
    #     (-50, 0),  # Translate -50 pixels along X-axis
    #     (0, -50),  # Translate -50 pixels along Y-axis
    ],
    "rotations": [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330],
    "scales": [
        # 0.5, 1.5, 2.0
    ],
    "shears": [
    #     (25, 0),  # Shear 25 degrees along X-axis
    #     (0, 25),  # Shear 25 degrees along Y-axis
    #     (-25, 0),  # Shear -25 degrees along X-axis
    #     (0, -25),  # Shear -25 degrees along Y-axis
    #     (50, 0),  # Shear 50 degrees along X-axis
    #     (0, 50),  # Shear 50 degrees along Y-axis
    #     (-50, 0),  # Shear -50 degrees along X-axis
    #     (0, -50),  # Shear -50 degrees along Y-axis
    ],
}

evaluate_architecture("vit_base", loader, geometric_transforms, improvement_type="TTA", n_augmentations=4)
evaluate_architecture("vit_base", loader, geometric_transforms, improvement_type="TTA", n_augmentations=8)
evaluate_architecture("vit_base", loader, geometric_transforms, improvement_type="TTA", n_augmentations=12)

Baseline (No Transformations):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [01:04<00:00,  1.23it/s]


Top-1 Accuracy: 71.40%
Top-5 Accuracy: 91.64%
Rotation 30:


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [01:10<00:00,  1.12it/s]


Top-1 Accuracy: 70.50%
Top-5 Accuracy: 90.86%
Rotation 60:


...

Top-1 Accuracy: 73.00%
Top-5 Accuracy: 92.48%
Rotation 300:


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [02:36<00:00,  1.99s/it]


Top-1 Accuracy: 70.46%
Top-5 Accuracy: 90.72%
Rotation 330:


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [02:37<00:00,  2.00s/it]

Top-1 Accuracy: 70.20%
Top-5 Accuracy: 90.70%


##### 2. Feature Averaging

In [29]:
geometric_transforms = {
    "baseline": True,
    "translations": [
    #     (25, 0),  # Translate 25 pixels along X-axis
    #     (0, 25),  # Translate 25 pixels along Y-axis
    #     (-25, 0),  # Translate -25 pixels along X-axis
    #     (0, -25),  # Translate -25 pixels along Y-axis
    #     (50, 0),  # Translate 50 pixels along X-axis
    #     (0, 50),  # Translate 50 pixels along Y-axis
    #     (-50, 0),  # Translate -50 pixels along X-axis
    #     (0, -50),  # Translate -50 pixels along Y-axis
    ],
    "rotations": [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330],
    "scales": [
        # 0.5, 1.5, 2.0
    ],
    "shears": [
    #     (25, 0),  # Shear 25 degrees along X-axis
    #     (0, 25),  # Shear 25 degrees along Y-axis
    #     (-25, 0),  # Shear -25 degrees along X-axis
    #     (0, -25),  # Shear -25 degrees along Y-axis
    #     (50, 0),  # Shear 50 degrees along X-axis
    #     (0, 50),  # Shear 50 degrees along Y-axis
    #     (-50, 0),  # Shear -50 degrees along X-axis
    #     (0, -50),  # Shear -50 degrees along Y-axis
    ],
}

evaluate_architecture("vit_base", loader, geometric_transforms, improvement_type="feature_averaging", n_augmentations=4)
evaluate_architecture("vit_base", loader, geometric_transforms, improvement_type="feature_averaging", n_augmentations=8)
evaluate_architecture("vit_base", loader, geometric_transforms, improvement_type="feature_averaging", n_augmentations=12)

Baseline (No Transformations):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [01:03<00:00,  1.24it/s]


Top-1 Accuracy: 71.40%
Top-5 Accuracy: 91.64%
Rotation 30:


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [01:05<00:00,  1.21it/s]


Top-1 Accuracy: 70.50%
Top-5 Accuracy: 90.86%
Rotation 60:


...

Top-1 Accuracy: 73.00%
Top-5 Accuracy: 92.48%
Rotation 300:


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [02:36<00:00,  1.98s/it]


Top-1 Accuracy: 70.46%
Top-5 Accuracy: 90.72%
Rotation 330:


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [02:37<00:00,  1.99s/it]

Top-1 Accuracy: 70.20%
Top-5 Accuracy: 90.70%


##### Canonicalization

In [14]:
geometric_transforms = {
    "baseline": True,
    "translations": [
    #     (25, 0),  # Translate 25 pixels along X-axis
    #     (0, 25),  # Translate 25 pixels along Y-axis
    #     (-25, 0),  # Translate -25 pixels along X-axis
    #     (0, -25),  # Translate -25 pixels along Y-axis
    #     (50, 0),  # Translate 50 pixels along X-axis
    #     (0, 50),  # Translate 50 pixels along Y-axis
    #     (-50, 0),  # Translate -50 pixels along X-axis
    #     (0, -50),  # Translate -50 pixels along Y-axis
    ],
    "rotations": [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330],
    "scales": [
        # 0.5, 1.5, 2.0
    ],
    "shears": [
    #     (25, 0),  # Shear 25 degrees along X-axis
    #     (0, 25),  # Shear 25 degrees along Y-axis
    #     (-25, 0),  # Shear -25 degrees along X-axis
    #     (0, -25),  # Shear -25 degrees along Y-axis
    #     (50, 0),  # Shear 50 degrees along X-axis
    #     (0, 50),  # Shear 50 degrees along Y-axis
    #     (-50, 0),  # Shear -50 degrees along X-axis
    #     (0, -50),  # Shear -50 degrees along Y-axis
    ],
}

evaluate_architecture("vit_base", loader, geometric_transforms, improvement_type="canonicalizer")

Baseline (No Transformations):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:21<00:00,  3.61it/s]


Top-1 Accuracy: 67.36%
Top-5 Accuracy: 87.44%
Rotation 30:


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:23<00:00,  3.32it/s]


Top-1 Accuracy: 65.86%
Top-5 Accuracy: 85.76%
Rotation 60:


...

Top-1 Accuracy: 64.98%
Top-5 Accuracy: 85.70%
Rotation 300:


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:22<00:00,  3.48it/s]


Top-1 Accuracy: 62.16%
Top-5 Accuracy: 83.68%
Rotation 330:


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:23<00:00,  3.32it/s]

Top-1 Accuracy: 63.16%
Top-5 Accuracy: 84.14%
